In [54]:
# Requriements file conf
# pip freeze > requirements.txt  --> To create, update requirements file
# pip install -r requirements.txt  --> To install all the packages from requirements file

# Import part
# Data manipulation and analysis
import pandas as pd
import numpy as np
import sklearn

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import LatentDirichletAllocation

# Natural Language Processing (NLP)
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# General utilities
import re
import matplotlib.pyplot as plt

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mehmet\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [55]:
# Read data
file_path = 'filtered_file_2.xlsx'
df = pd.read_excel(file_path)

#print(data.head(10))
print(df.info())
print(df.describe())
print(df.isnull().sum())

# Print column names
print(df.columns)

# Download stopwords and punkt
nltk.download('punkt')
nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17753 entries, 0 to 17752
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Article Title       17752 non-null  object
 1   Abstract            17753 non-null  object
 2   Researcher Ids      17753 non-null  object
 3   ORCIDs              14138 non-null  object
 4   DOI                 17753 non-null  object
 5   WoS Categories      17752 non-null  object
 6   Research Areas      17752 non-null  object
 7   IDS Number          17753 non-null  object
 8   UT (Unique WOS ID)  17753 non-null  object
 9   Publication Year    17753 non-null  int64 
dtypes: int64(1), object(9)
memory usage: 1.4+ MB
None
       Publication Year
count      17753.000000
mean        2015.791585
std            6.559040
min         1986.000000
25%         2012.000000
50%         2017.000000
75%         2021.000000
max         2025.000000
Article Title            1
Abstract         

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mehmet\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mehmet\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [56]:
# Remove special characters, punctuation, and convert to lowercase
def preprocess_text(text):
    if not isinstance(text, str):
        return ''  # Return empty string for NaN or non-string values
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and convert to lowercase
    text = ' '.join(word for word in text.split() if word not in STOPWORDS)  # Remove stopwords
    return text

df['Abstract_clean'] = df['Abstract'].apply(preprocess_text)
df['Research_Areas_clean'] = df['Research Areas'].apply(preprocess_text)

In [57]:
# Print the cleaned text
print(df['Abstract_clean'].head())
print(df['Research_Areas_clean'].head())

0    latent heat storage system two different stack...
1    attempt investigate effect testicular torsion ...
2    longlived fission radionuclides spread chernob...
3    prostheses proposed restore spinal stability p...
4    vertical particle flux measured two sites sout...
Name: Abstract_clean, dtype: object
0                            energy fuels
1                      urology nephrology
2    chemistry nuclear science technology
3                  biophysics engineering
4                            oceanography
Name: Research_Areas_clean, dtype: object


In [58]:
# Function to split multiple researcher IDs into individual rows
def split_researchers(row):
    if not isinstance(row['Researcher Ids'], str):
        return pd.DataFrame()  # Skip invalid entries
    researchers = row['Researcher Ids'].split(';')
    data = []
    for researcher_id in researchers:
        data.append({
            'Researcher': researcher_id.strip(),
            'Abstract_clean': row['Abstract_clean'],
            'Research_Areas_clean': row['Research_Areas_clean'],
            'Publication Year': row['Publication Year']
        })
    return pd.DataFrame(data)

# Flatten the dataset
flattened_df = pd.concat(df.apply(split_researchers, axis=1).to_list(), ignore_index=True)

# Display the flattened dataset
flattened_df.head()

,Researcher,Abstract_clean,Research_Areas_clean,Publication Year
0,"Cengel, Yunus/D-4105-2015",latent heat storage system two different stack...,energy fuels,1986
1,"Canda, M.Serefittin/K-4298-2019",attempt investigate effect testicular torsion ...,urology nephrology,1988
2,"AKCAY, HUSEYIN/ABI-8239-2020",longlived fission radionuclides spread chernob...,chemistry nuclear science technology,1988
3,"Alıcı Davutoğlu, Ebru/AIE-5610-2022",prostheses proposed restore spinal stability p...,biophysics engineering,1990
4,"Ittekkot, Venugopalan/AAC-3437-2022",vertical particle flux measured two sites sout...,oceanography,1990


In [34]:
# Group by researcher and aggregate their data
grouped_df = flattened_df.groupby('Researcher').agg({
    'Abstract_clean': ' '.join,  # Combine all abstracts
    'Research_Areas_clean': ' '.join,  # Combine all research areas
    'Publication Year': 'max'  # Use the max publication year for weighting
}).reset_index()

# Add a combined text column for analysis
grouped_df['combined_text'] = grouped_df['Abstract_clean'] + ' ' + grouped_df['Research_Areas_clean']

# Display the grouped data
grouped_df.head()

,Researcher,Abstract_clean,Research_Areas_clean,Publication Year,combined_text
0,,damage potential columns building frames due e...,construction building technology engineering u...,2024,damage potential columns building frames due e...
1,(data truncated to fit),2008 published first set guidelines standardiz...,cell biology,2021,2008 published first set guidelines standardiz...
2,", 0000-0002-0233-0025/F-6037-2011",adsorption copper zinc cobalt lead cadmium ion...,biochemistry molecular biology environmental s...,2012,adsorption copper zinc cobalt lead cadmium ion...
3,", Abdullah/V-9765-2019",coal produced underground mines transported su...,metallurgy metallurgical engineering mining mi...,2023,coal produced underground mines transported su...
4,", Alireza/AAE-4134-2022",high blood cholesterol typically considered fe...,science technology topics life sciences biomed...,2023,high blood cholesterol typically considered fe...


In [35]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=1000, sublinear_tf=True, max_df=0.8, min_df=2)
tfidf_matrix = tfidf.fit_transform(grouped_df['combined_text'])

# Display the shape of the TF-IDF matrix
print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF matrix shape: (20320, 1000)


In [43]:
# User project idea
user_input = "cell cancer bio bioinformatics machine learning algorithm human life gene gen heredity"
user_input_clean = preprocess_text(user_input)

# Transform the user's input into a TF-IDF vector
user_vector = tfidf.transform([user_input_clean])

# Compute similarity scores
similarity_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

# Add similarity scores to the grouped DataFrame
grouped_df['similarity'] = similarity_scores

In [50]:
# Count the number of articles per researcher
article_counts = flattened_df['Researcher'].value_counts()
grouped_df['article_count'] = grouped_df['Researcher'].map(article_counts)

# Normalize similarity scores based on the number of articles
k = 0.184  # Growth rate, calculated to make f(25) approximately 1
grouped_df['normalized_similarity'] = grouped_df['similarity'] * (1 - np.exp(-k * grouped_df['article_count']))

# Scale year weights for stronger impact (higher weight for recent years)
# Year set to 2026 because the articles published 2025 are being 2026 in the dataset
current_year = 2026
year_diff = current_year - grouped_df['Publication Year']
a=1
grouped_df['year_weight'] =  0.75 + 0.2 * (1 - 1 / (1 + np.exp(-a * (year_diff - 5))))

# Combine normalized similarity and year weight into a final score
grouped_df['final_score'] = grouped_df['normalized_similarity'] * grouped_df['year_weight']


In [52]:
# Sort by the final score
top_matches = grouped_df.sort_values(by='final_score', ascending=False)

# Display top matches
print("Top Recommended Collaborators:")
print(top_matches[['Researcher', 'final_score', 'article_count', 'year_weight', 'Publication Year']])

# Case-insensitive search
specific_researcher = top_matches[top_matches['Researcher'].str.contains('ABG-9392-2021', case=False, na=False)]

# Print the result
print(specific_researcher[['Researcher', 'final_score', 'article_count', 'year_weight', 'Publication Year']])

Top Recommended Collaborators:
                                  Researcher  final_score  article_count  \
7951              Işık, Zerrin/ABG-9392-2021     0.198576             22   
8490      Kalayci Demir, Guleser/O-7249-2019     0.191539             20   
10019          Leblebici, Asım/HHM-5895-2022     0.182631             14   
2627       Birant, Kökten Ulaş/AAD-8386-2022     0.180936             16   
6463   Georgakilas, Alexandros/AAG-9517-2020     0.163145             20   
...                                      ...          ...            ...   
25                     , Osman/AGR-7980-2022     0.000000              1   
24                      , OMER/AAZ-9186-2020     0.000000              1   
23                 , Muge Gunalp/S-1323-2017     0.000000              1   
21                  , Ljiljana/AAY-7289-2020     0.000000              1   
9                       , Daurim/R-5825-2019     0.000000              1   

       year_weight  Publication Year  
7951      0.94051